[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/06_rnn/06_rnn_solutions.ipynb)

# 06. RNN — 연습 문제 해설

[06_rnn.ipynb](06_rnn.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch matplotlib numpy

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 연습 1. Char-RNN — hidden_size를 2~3으로 줄이면?

In [ ]:
chars = list("hielo")
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}

text = "hihello"
x_str, y_str = text[:-1], text[1:]
x_idx = [char2idx[c] for c in x_str]
y_idx = [char2idx[c] for c in y_str]
n_classes = len(chars)

X = torch.tensor(np.eye(n_classes)[x_idx], dtype=torch.float32).unsqueeze(0).to(device)
Y = torch.tensor(y_idx, dtype=torch.long).unsqueeze(0).to(device)

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out)

def train_char_rnn(hidden_size, epochs=200, lr=0.1):
    torch.manual_seed(0)
    model = CharRNN(n_classes, hidden_size, n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(X)
        loss = loss_fn(out.view(-1, n_classes), Y.view(-1))
        loss.backward()
        optimizer.step()
    pred = out.argmax(dim=2).squeeze(0).cpu().numpy()
    pred_str = "".join(idx2char[i] for i in pred)
    return loss.item(), pred_str

for h in [2, 3, 8]:
    loss, pred_str = train_char_rnn(hidden_size=h)
    correct = "✅" if pred_str == "ihello" else "❌"
    print(f"hidden_size={h:<3} 최종 loss={loss:.4f}  예측='{pred_str}'  목표='ihello' {correct}")

**해설**
- `"hihell" -> "ihello"`를 맞히려면, RNN은 같은 입력 글자 `'h'` 뒤에도 문맥에 따라 다른 다음 글자(처음 `h` 다음엔 `i`, 이후 `h` 다음엔 여전히 `i`지만 두 번째 `l` 다음엔 `l`이 아니라 `o`)를 구분해서 예측해야 합니다. 즉 **바로 직전 글자만으로는 부족하고, 이전 문맥(은닉 상태)을 기억**해야 풉니다.
- `hidden_size`가 너무 작으면(2~3) 이 문맥 정보를 담을 "용량"이 부족해서 학습이 불안정하거나 특정 위치에서 계속 틀립니다.
- `hidden_size=8` 정도면 이 짧은 시퀀스의 문맥을 담기에 충분해서 안정적으로 `"ihello"`를 맞힙니다.
- 결론: 은닉 상태 크기는 "모델이 기억해야 하는 문맥의 복잡도"에 맞춰 정해야 합니다 — 04번 노트북의 XOR 은닉 유닛 실험과 같은 맥락(용량 부족 → 학습 실패/불안정)입니다.

## 연습 2. Sine 시계열 — RNN vs LSTM vs GRU

In [ ]:
t = np.linspace(0, 100, 1000)
series = np.sin(t)
SEQ_LEN = 20

def make_sequences(series, seq_len):
    xs, ys = [], []
    for i in range(len(series) - seq_len):
        xs.append(series[i:i + seq_len])
        ys.append(series[i + seq_len])
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(series, SEQ_LEN)
split = int(len(X_seq) * 0.8)
X_train = torch.tensor(X_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)
y_train = torch.tensor(y_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)
X_test = torch.tensor(X_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)
y_test = torch.tensor(y_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)

In [ ]:
class TimeSeriesModel(nn.Module):
    def __init__(self, cell_cls, hidden_size=16):
        super().__init__()
        self.rnn = cell_cls(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

def train_and_eval(cell_cls, epochs=150, lr=0.01):
    torch.manual_seed(0)
    model = TimeSeriesModel(cell_cls).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(X_test), y_test).item()
    return model, test_mse

cells = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
models, mses = {}, {}
for name, cell_cls in cells.items():
    model, test_mse = train_and_eval(cell_cls)
    models[name] = model
    mses[name] = test_mse
    print(f"{name:<5} test MSE = {test_mse:.6f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test.cpu().numpy().flatten(), label="실제값", linewidth=2)
for name, model in models.items():
    with torch.no_grad():
        pred = model(X_test).cpu().numpy().flatten()
    plt.plot(pred, "--", label=f"{name} 예측 (MSE={mses[name]:.5f})")
plt.title("RNN vs LSTM vs GRU — Sine 시계열 예측")
plt.legend()
plt.show()

**해설**
- `nn.RNN`을 `nn.LSTM`/`nn.GRU`로 바꾸는 것은 인터페이스상으로는 클래스 이름만 바꾸면 될 정도로 간단합니다 (입출력 shape 동일).
- 이 sine 곡선처럼 비교적 짧고 규칙적인 시퀀스(`SEQ_LEN=20`)에서는 기본 RNN도 이미 잘 작동해서 셋의 성능 차이가 크지 않을 수 있습니다.
- LSTM/GRU는 **게이트(gate) 구조**로 어떤 정보를 기억하고 잊을지 학습하기 때문에, 시퀀스가 훨씬 길어지거나 장기 의존성(long-term dependency)이 중요한 문제일수록 기본 RNN보다 확실히 유리해집니다 — Vanishing Gradient 문제에 더 강건합니다.
- 실무에서는 특별한 이유가 없다면 기본 `nn.RNN`보다 `nn.GRU`(가볍고 성능 좋음) 또는 `nn.LSTM`(가장 널리 검증됨)을 기본 선택지로 삼는 경우가 많습니다.